# Event displays: get_flavour vs. IFN, quark jet label from the shower history

`JetTaggingUtils::get_flavour` labels a jet with the heaviest quark within 0.3 rad of the jet axis, using every
status-2 quark of the event record (the Delphes samples only have status 1 and 2). That includes the quarks between
the matrix element and the hadronisation, e.g. the b directly from H → bb̄. If that b radiates a hard gluon, the gluon
can form its own jet while the b that finally enters the hadronisation ends up in the other jet. The
gluon jet then still gets the label b from the original b direction, while IFN on the hadronisation partons finds no
flavour in it (`g | g`).

This notebook (parton level, exclusive 2-jet ee_kt + IFN, same as `jet-event-display.ipynb`) selects events with a jet
that `get_flavour` labels as the Higgs decay flavour q although none of the jet's hadronisation partons is a q or q̄,
and draws them in the plane of the two jets:

* x axis: direction of jet 1, y axis: component of jet 2 orthogonal to jet 1.
* Thin arrows: hadronisation partons, coloured by the IFN jet they are clustered into (bold label: quarks).
  Dashed: mostly out of the plane (p_T > p_L w.r.t. the jet plane).
* Thick translucent arrows: jet axes; dotted lines: the 0.3 rad `get_flavour` matching cone around each jet (in the plane).
* Thick black arrows: the q and q̄ from the Higgs decay (`q_{H}`). Thick green arrows: gluons radiated off the q and q̄
  shower chains with |p| > `EMISSION_MIN` (the hardest one is always drawn).
* The inset in the top right corner zooms into the centre (factor `ZOOM`), where the soft partons are; arrows longer
  than its range are cut at the edge.
* The legend gives the IFN content and the `get_flavour` label of each jet, the true 3D angles of q_{H} and q̄_{H} to
  the jets, and in which jet the q / q̄ at the end of the chain (entering hadronisation) ends up.

Events are drawn in order of the hardest radiated gluon (clearest examples first). Plots:
`outputs/jet-test/event_displays/parton_getflav_mismatch/<sample>/entry_<n>.pdf`, all of them combined in
`all_events.pdf`, and a `README.md`.

Usage as for `jet-event-display.ipynb`: `SAMPLE=<sample> jupyter nbconvert --to notebook --execute --output <copy>.ipynb jet-getflav-mismatch.ipynb`
(H → qq̄ samples only; default Hbb).

In [1]:
import os
import re
import glob
import shutil

import numpy as np
import ROOT
from particle import Particle

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
# see jet-event-display.ipynb: avoid the LoadPCM flood with a stale build, load libFastJet first
ROOT.gErrorIgnoreLevel = ROOT.kWarning
ROOT.gSystem.Load("libFastJet")
ROOT.gSystem.Load("libFCCAnalyses")
ROOT.dummyLoader()

True

In [2]:
SAMPLE = os.environ.get("SAMPLE", "wzp6_ee_nunuH_Hbb_ecm240")
INPUT_DIR = f"/eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/{SAMPLE}"
N_FILES = 1          # number of input files to chain
N_EVENTS = 20000     # events to scan, -1 for all events in the chained files
N_DRAW = 10          # number of events to draw
EMISSION_MIN = 10.   # draw radiated gluons with |p| above this (GeV)
MATCH_ANGLE = 0.3    # get_flavour matching angle (rad)

FLAVOURS = ["u", "d", "s", "c", "b"]
PDG = {"d": 1, "u": 2, "s": 3, "c": 4, "b": 5}
Q = SAMPLE.split("_H")[1][0]
if Q not in PDG:
    raise ValueError(f"{SAMPLE}: only H -> q qbar samples are supported")
Q_PDG = PDG[Q]

PLOT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test", "event_displays", "parton_getflav_mismatch", SAMPLE)
if os.path.isdir(PLOT_DIR):
    shutil.rmtree(PLOT_DIR)  # remove the displays of a previous run
os.makedirs(PLOT_DIR)

In [3]:
ROOT.gInterpreter.Declare("""
// 1 for jets that get_flavour labels as q although none of their constituents (hadronisation partons) is a q or qbar
ROOT::VecOps::RVec<int> getflav_mismatch(const ROOT::VecOps::RVec<int>& label,
                                         const std::vector<std::vector<int>>& constituents,
                                         const ROOT::VecOps::RVec<float>& pdg, int q) {
  ROOT::VecOps::RVec<int> r(label.size(), 0);
  for (size_t j = 0; j < label.size(); ++j) {
    if (label[j] != q) continue;
    bool has_q = false;
    for (int i : constituents[j]) has_q |= std::abs(pdg[i]) == q;
    r[j] = !has_q;
  }
  return r;
}""")

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))[:N_FILES]
chain = ROOT.TChain("events")
for f in files:
    chain.Add(f)

df = ROOT.RDataFrame(chain)
if N_EVENTS > 0:
    df = df.Range(N_EVENTS)
df = (df.Define("entry", "rdfentry_")
        .Define("partons", "FCCAnalyses::MCParticle::get_hadronisation_partons(Particle, _Particle_parents.index)")
        .Define("parton_px", "FCCAnalyses::MCParticle::get_px(partons)")
        .Define("parton_py", "FCCAnalyses::MCParticle::get_py(partons)")
        .Define("parton_pz", "FCCAnalyses::MCParticle::get_pz(partons)")
        .Define("parton_m", "FCCAnalyses::MCParticle::get_mass(partons)")
        .Define("parton_pdg", "FCCAnalyses::MCParticle::get_pdg(partons)")
        .Define("pseudo_jets", "FCCAnalyses::JetClusteringUtils::set_pseudoJets_xyzm_flav(parton_px, parton_py, parton_pz, parton_m, parton_pdg)")
        .Define("clustered_jets", "JetClustering::flv_ifn_clustering_ee_kt(2, 2, 0, 0)(pseudo_jets)")
        .Define("jets", "FCCAnalyses::JetClusteringUtils::get_pseudoJets(clustered_jets)")
        .Define("jets_constituents", "FCCAnalyses::JetClusteringUtils::get_constituents(clustered_jets)")
        .Define("jets_getflav", "FCCAnalyses::JetTaggingUtils::get_flavour(jets, Particle)")
        .Define("jets_mismatch", f"getflav_mismatch(jets_getflav, jets_constituents, parton_pdg, {Q_PDG})")
        # full event record, for the shower history
        .Define("mc_pdg", "Particle.PDG")
        .Define("mc_px", "Particle.momentum.x")
        .Define("mc_py", "Particle.momentum.y")
        .Define("mc_pz", "Particle.momentum.z")
        .Define("mc_parents_begin", "Particle.parents_begin")
        .Define("mc_parents_end", "Particle.parents_end")
        .Define("mc_parents", "_Particle_parents.index"))
for var in ["px", "py", "pz", "e"]:
    df = df.Define(f"jets_{var}", f"FCCAnalyses::JetClusteringUtils::get_{var}(jets)")
for f in FLAVOURS:
    df = df.Define(f"jets_{f}_content", f"FCCAnalyses::JetClusteringUtils::get_{f}_content(jets)")

n_total = df.Count()
n_two = df.Filter("jets.size() == 2").Count()
n_neutral = df.Filter(f"jets.size() == 2 && jets_{Q}_content[0] == 0").Count()
columns = (["entry", "parton_px", "parton_py", "parton_pz", "parton_pdg", "jets_constituents", "jets_getflav",
            "jets_mismatch", "mc_pdg", "mc_px", "mc_py", "mc_pz", "mc_parents_begin", "mc_parents_end", "mc_parents"]
           + [f"jets_{v}" for v in ["px", "py", "pz", "e"]] + [f"jets_{f}_content" for f in FLAVOURS])
events = df.Filter("jets.size() == 2 && Sum(jets_mismatch) > 0", "get_flavour mismatch").AsNumpy(columns)
n_sel = len(events["entry"])
print(f"{SAMPLE}: {n_sel} of {n_two.GetValue()} two-jet events ({n_sel / n_two.GetValue():.2%}) have a jet labelled "
      f"{Q} by get_flavour without any {Q} / {Q}bar hadronisation parton; {n_neutral.GetValue()} events are "
      f"{Q} content 0 for IFN")

#--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                                       
# FastJet is provided without warranty under the GNU GPL v2 or higher.  
# It uses T. Chan's closest pair algorithm, S. Fortune's Voronoi code
# and 3rd party plugin jet algorithms. See COPYING file for details.
#--------------------------------------------------------------------------


wzp6_ee_nunuH_Hbb_ecm240: 199 of 20000 two-jet events (1.00%) have a jet labelled b by get_flavour without any b / bbar hadronisation parton; 866 events are b content 0 for IFN


In [4]:
def angle(a, b):
    return float(np.arccos(np.clip(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)), -1., 1.)))


def shower_history(i_ev):
    """q and qbar from the Higgs decay, gluons radiated off their shower chains, and the end of each chain.

    The chain follows the same-PDG daughter (daughters derived from the reliable parents links) as long as there is
    exactly one; gluons among the daughters of chain members are the emissions.
    """
    pdg = np.asarray(events["mc_pdg"][i_ev])
    mom = np.array([np.asarray(events[f"mc_{v}"][i_ev]) for v in ["px", "py", "pz"]]).T
    begin, end = np.asarray(events["mc_parents_begin"][i_ev]), np.asarray(events["mc_parents_end"][i_ev])
    parents_idx = np.asarray(events["mc_parents"][i_ev])
    parents = [parents_idx[begin[i]:end[i]] for i in range(len(pdg))]
    children = {}
    for i, ps in enumerate(parents):
        for p in ps:
            children.setdefault(int(p), []).append(i)
    history = []
    for i in range(len(pdg)):
        if abs(pdg[i]) != Q_PDG or not any(pdg[p] == 25 for p in parents[i]):
            continue
        cur, emissions = i, []
        while True:
            ch = children.get(cur, [])
            emissions += [mom[c] for c in ch if pdg[c] == 21]
            same = [c for c in ch if pdg[c] == pdg[i]]
            if len(same) != 1:
                break
            cur = same[0]
        history.append(dict(pdg=int(pdg[i]), p=mom[i], emissions=emissions, end=mom[cur]))
    return history


def hardest_emission(i_ev):
    return max((np.linalg.norm(e) for h in shower_history(i_ev) for e in h["emissions"]), default=0.)

In [5]:
JET_COLOURS = [ROOT.kRed + 1, ROOT.kAzure + 2]
EMISSION_COLOUR = ROOT.kGreen + 2
ZOOM = 6  # the inset shows the central 1 / ZOOM of the main plot
PARTON_NAMES = {1: "d", 2: "u", 3: "s", 4: "c", 5: "b", 6: "t", 21: "g"}


def parton_label(pdg):
    name = PARTON_NAMES.get(abs(pdg), str(pdg))
    return f"#bar{{{name}}}" if pdg < 0 and abs(pdg) <= 6 else name


def jet_plane(j1, j2):
    """Orthonormal basis (e1, e2, n) with e1 along jet 1 and jet 2 in the (e1, e2) plane."""
    e1 = j1 / np.linalg.norm(j1)
    v = j2 - j2.dot(e1) * e1
    if np.linalg.norm(v) < 1e-6 * np.linalg.norm(j2):
        v = np.cross(e1, [0., 0., 1.] if abs(e1[2]) < 0.9 else [1., 0., 0.])
    e2 = v / np.linalg.norm(v)
    return e1, e2, np.cross(e1, e2)


def event_location(entry):
    chain.LoadTree(int(entry))
    return chain.GetFile().GetName(), int(entry) - chain.GetChainOffset()


def draw_event(i_ev, jets, partons, pdgs, jet_of, history, proj, out_of_plane, r, text_scale, keep):
    """Draw the event into the current pad, for the range [-r, r]; arrows longer than that are cut at the edge."""
    jx, jy = zip(*(proj(j) for j in jets))

    def clip(x, y):
        s = max(abs(x), abs(y)) / (0.98 * r)
        return (x / s, y / s) if s > 1 else (x, y)

    def arrow(p, colour, width, size, alpha=1.):
        x, y = clip(*proj(p))
        a = ROOT.TArrow(0., 0., x, y, size, "|>")
        a.SetLineWidth(width)
        a.SetLineColorAlpha(colour, alpha)
        a.SetFillColorAlpha(colour, alpha)
        if out_of_plane(p):
            a.SetLineStyle(2)
        a.Draw()
        keep.append(a)
        return x, y

    def label(x, y, text, colour, size, bold, align=None):
        t = ROOT.TLatex(x * 1.04, y * 1.04, text)
        t.SetTextAlign(align or (31 if x < 0 else 11) + (0 if y < 0 else 2) - 1)  # away from the origin
        t.SetTextSize(size * text_scale)
        t.SetTextFont(62 if bold else 42)
        t.SetTextColor(colour)
        t.Draw()
        keep.append(t)

    for i_jet in range(2):  # jet axes and get_flavour matching cones (in the plane)
        arrow(jets[i_jet], JET_COLOURS[i_jet], 8, 0.025, alpha=0.3)
        phi = np.arctan2(jy[i_jet], jx[i_jet])
        for s in (-1, 1):
            x, y = clip(2 * r * np.cos(phi + s * MATCH_ANGLE), 2 * r * np.sin(phi + s * MATCH_ANGLE))
            line = ROOT.TLine(0., 0., x, y)
            line.SetLineColor(JET_COLOURS[i_jet])
            line.SetLineStyle(3)
            line.Draw()
            keep.append(line)

    for p, pdg, i_jet in zip(partons, pdgs, jet_of):
        colour = JET_COLOURS[i_jet] if i_jet >= 0 else ROOT.kGray + 2
        x, y = arrow(p, colour, 2, 0.012)
        is_quark = abs(pdg) <= 6
        label(x, y, parton_label(pdg), colour, 0.032 if is_quark else 0.022, is_quark)

    hardest = max((np.linalg.norm(e) for h in history for e in h["emissions"]), default=0.)
    for h in history:
        for e in h["emissions"]:
            if np.linalg.norm(e) > EMISSION_MIN or np.linalg.norm(e) == hardest:
                x, y = arrow(e, EMISSION_COLOUR, 4, 0.02)
                # label at the middle of the arrow, below / above it, so that it does not cover the q_H label
                label(0.5 * x, 0.5 * y, f"g ({np.linalg.norm(e):.0f} GeV, from {parton_label(h['pdg'])}_{{H}})",
                      EMISSION_COLOUR, 0.026, True, align=23 if y <= 0 else 21)
        x, y = arrow(h["p"], ROOT.kBlack, 4, 0.02)
        label(x, y, f"{parton_label(h['pdg'])}_{{H}}", ROOT.kBlack, 0.036, True)


def event_display(i_ev):
    entry = int(events["entry"][i_ev])
    jets = [np.array([events[f"jets_{v}"][i_ev][i] for v in ["px", "py", "pz"]]) for i in range(2)]
    partons = np.array([np.asarray(events[f"parton_{v}"][i_ev]) for v in ["px", "py", "pz"]]).T
    pdgs = np.asarray(events["parton_pdg"][i_ev]).astype(int)
    jet_of = np.full(len(pdgs), -1)
    for i_jet, constituents in enumerate(events["jets_constituents"][i_ev]):
        jet_of[np.asarray(constituents)] = i_jet
    history = shower_history(i_ev)

    e1, e2, n = jet_plane(*jets)
    proj = lambda p: (float(p @ e1), float(p @ e2))
    out_of_plane = lambda p: abs(p @ n) > np.hypot(*proj(p))
    coords = [c for p in list(partons) + jets + [h["p"] for h in history] for c in proj(p)]
    r = 1.15 * max(np.max(np.abs(coords)), 1.)

    c = ROOT.TCanvas(f"evt_{SAMPLE}_{entry}", "", 800, 800)
    c.SetMargin(0.12, 0.05, 0.1, 0.1)
    frame = c.DrawFrame(-r, -r, r, r)
    frame.SetTitle(f"{SAMPLE}, parton level, entry {entry};p #upoint #hat{{e}}_{{1}} (jet 1) [GeV];p #upoint #hat{{e}}_{{2}} [GeV]")
    keep = [frame]
    draw_event(i_ev, jets, partons, pdgs, jet_of, history, proj, out_of_plane, r, 1., keep)

    leg = ROOT.TLegend(0.13, 0.11, 0.75, 0.33)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextSize(0.021)
    for i_jet in range(2):
        line = ROOT.TLine()
        line.SetLineColor(JET_COLOURS[i_jet])
        line.SetLineWidth(3)
        content = " ".join(f"{f}{int(events[f'jets_{f}_content'][i_ev][i_jet]):+d}" for f in FLAVOURS)
        gf = int(events["jets_getflav"][i_ev][i_jet])
        gf = PARTON_NAMES.get(gf, "unmatched") if gf else "unmatched"
        leg.AddEntry(line, f"jet {i_jet + 1} ({events['jets_e'][i_ev][i_jet]:.0f} GeV): IFN {content};  get_flavour: {gf}", "l")
        keep.append(line)
    for h in history:
        name = parton_label(h["pdg"])
        # jet of the chain end = jet of the hadronisation parton with the same momentum
        match = [i for i, p in enumerate(partons) if pdgs[i] == h["pdg"] and np.allclose(p, h["end"], rtol=1e-4, atol=1e-3)]
        end_jet = f"jet {jet_of[match[0]] + 1}" if match else "not a hadronisation parton"
        black = ROOT.TLine()
        black.SetLineWidth(4)
        leg.AddEntry(black, f"{name}_{{H}} ({np.linalg.norm(h['p']):.0f} GeV): #angle to jet 1 / 2 = "
                     f"{angle(h['p'], jets[0]):.2f} / {angle(h['p'], jets[1]):.2f} rad;  {name} at hadronisation "
                     f"({np.linalg.norm(h['end']):.0f} GeV) in {end_jet}", "l")
        keep.append(black)
    dotted = ROOT.TLine()
    dotted.SetLineStyle(3)
    leg.AddEntry(dotted, f"get_flavour matching cone ({MATCH_ANGLE} rad, in the plane)", "l")
    dashed = ROOT.TLine()
    dashed.SetLineStyle(2)
    leg.AddEntry(dashed, "p_{T} > p_{L}  (w.r.t. jet plane)", "l")
    leg.Draw()
    keep += [leg, dotted, dashed]

    # inset: zoom into the centre, where the soft partons (often the quark at the end of the chain) are
    inset = ROOT.TPad(f"inset_{entry}", "", 0.62, 0.62, 0.95, 0.9)
    inset.SetMargin(0.12, 0.04, 0.1, 0.04)
    inset.Draw()
    inset.cd()
    rz = r / ZOOM
    iframe = inset.DrawFrame(-rz, -rz, rz, rz)
    iframe.GetXaxis().SetLabelSize(0.07)
    iframe.GetYaxis().SetLabelSize(0.07)
    iframe.GetXaxis().SetNdivisions(505)
    iframe.GetYaxis().SetNdivisions(505)
    draw_event(i_ev, jets, partons, pdgs, jet_of, history, proj, out_of_plane, rz, 2.2, keep)
    zl = ROOT.TLatex(-0.95 * rz, 0.85 * rz, f"#times{ZOOM}")
    zl.SetTextSize(0.08)
    zl.Draw()
    keep += [inset, iframe, zl]
    c.cd()

    c._keep = keep
    return c

In [6]:
order = sorted(range(n_sel), key=lambda i: -hardest_emission(i))
canvases = []
readme = [f"Parton-level IFN jets in {SAMPLE}: events where `JetTaggingUtils::get_flavour` labels a jet as {Q} although "
          f"none of its hadronisation partons is a {Q} or {Q}bar ({n_sel} of {n_two.GetValue()} two-jet events). "
          f"The {N_DRAW} events with the hardest gluon radiated off the {Q} / {Q}bar shower chains are drawn "
          f"(`entry_<n>.pdf`, combined in `all_events.pdf`). Generated by "
          f"`examples/FCCee/flav_aware_jets/jet-getflav-mismatch.ipynb`.", ""]
for i_ev in order[:N_DRAW]:
    entry = int(events["entry"][i_ev])
    fname, local_entry = event_location(entry)
    print(f"entry {entry} (hardest emission {hardest_emission(i_ev):.1f} GeV)  ->  "
          f"edm4hep2json -e {local_entry} -l Particle -o out.json {fname}")
    readme.append(f"- `entry_{entry}.pdf`: hardest emission {hardest_emission(i_ev):.1f} GeV; "
                  f"`edm4hep2json -e {local_entry} -l Particle -o out.json {fname}`")
    c = event_display(i_ev)
    c.SaveAs(os.path.join(PLOT_DIR, f"entry_{entry}.pdf"))
    c.Draw()
    canvases.append(c)

combined = os.path.join(PLOT_DIR, "all_events.pdf")
canvases[0].Print(combined + "[")
for c in canvases:
    c.Print(combined, f"Title:{c.GetName()}")
canvases[-1].Print(combined + "]")
with open(os.path.join(PLOT_DIR, "README.md"), "w") as f:
    f.write("\n".join(readme) + "\n")

entry 2997 (hardest emission 85.7 GeV)  ->  edm4hep2json -e 2997 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 8279 (hardest emission 79.2 GeV)  ->  edm4hep2json -e 8279 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 11203 (hardest emission 78.8 GeV)  ->  edm4hep2json -e 11203 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 5509 (hardest emission 77.0 GeV)  ->  edm4hep2json -e 5509 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 2339 (hardest emission 76.9 GeV)  ->  ed

entry 12875 (hardest emission 64.3 GeV)  ->  edm4hep2json -e 12875 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
